In [78]:
import sys
sys.path.append('/Users/brianlerner/Library/CloudStorage/OneDrive-Personal/Code/cerebra/src')
from server.query import get_schema, get_events
from utils.server import connect_to_collection
from utils.general import to_snake_case

In [60]:
collection = connect_to_collection()

In [89]:
import prompts.bio as PROMPTS

Let's walkthrough the logic behind the analysis pipeline.

Let's first consider a user request: `How many days per week have I practice?`

In order to get a response for the user, the following needs to be known for initial data extraction:
- name: practice
- start_date: not specified
- end_date: not specified
- start_time: not specified
- end_time: not specified
<!-- - operation:  -->

The first issue that needs to be overcome is the issue of the label. In order to overcome this, we'll find the current labels available and see what is similar.

In [34]:
schema = get_schema()
variable_names = list(schema.keys())
print("Current variable names:", variable_names)

Current variable names: ['elbow pain', 'Basketball Game', 'Computer Programming', 'Basketball Practice', 'Discrete Mathematics', 'Discrete Mathematics HW due', 'Computer Programming HW due']


In [52]:
from openai import OpenAI
client = OpenAI()

variable_info = {
        # 'name': 'practice'
        'name': 'bball practice'
    }

def get_embedding(text, model="text-embedding-ada-002"):
   text = text.replace("\n", " ")
   return client.embeddings.create(input = [text], model=model).data[0].embedding

def get_closest_variable_names(base_name, variable_names, top_n=-1, threshold=0.05, return_list=True):
    import pandas as pd
    from scipy.spatial.distance import cosine 


    variable_names = [base_name] + variable_names

    df = pd.DataFrame(variable_names, columns=["name"])

    # this will be done beforehand during real model
    df["embeddings"] = df.name.apply(lambda x: get_embedding(x, model='text-embedding-ada-002'))

    df["distances"] = df["embeddings"].apply(lambda x: cosine(x, df.iloc[0]["embeddings"])) 
    df.sort_values(by="distances", inplace=True)

    if threshold > 0:
        df = df[df["distances"] <= threshold]

    df.drop("embeddings", axis=1, inplace=True)
    df = df.iloc[1:]
    if top_n > 0:
        df = df.iloc[0:top_n+1]

    if return_list:
        return list(df.name)
    else:
        return df

closest_variable_names = get_closest_variable_names(variable_info["name"], variable_names, top_n=1, threshold=0.05)

if len(closest_variable_names) == 1:
    closest_variable_name = closest_variable_names[0]
    print("Closest variable name:", closest_variable_name)
elif len(closest_variable_names) > 1:
    print("Multiple variables found. Please select one:", closest_variable_names)
elif len(closest_variable_names) == 0:
    print("No variables found. Please try again.")




Closest variable name: Basketball Practice


Now, let's get events using

In [62]:
conditions = {
    "start_date":None,
    "end_date":None,
    "start_time":None,
    "end_time":None,
    }


def get_events(
    name=None,
    start_date=None, 
    end_date=None, 
    start_time=None,
    end_time=None,
    # cat=None,
    # group=None, 
    profile='llm_v0', 
    print_events=False
    ):
    

    schema = get_schema()
    if name in schema:
        cat = schema[name]
    else:
        print(f'name: {name} not found in schema')
        return []
    # conditions = [
    #                 {"$gte": ["$$event.start_datetime", start_date]},
    #                 {"$lte": ["$$event.start_datetime", end_date]},
    #             ]
    conditions = []
    if start_date:
        conditions.append({"$gte": ["$$event.start_datetime", start_date]})
    if end_date:
        conditions.append({"$lte": ["$$event.start_datetime", end_date]})
    if name:
        conditions.append({"$eq": ["$$event.name", name]})
    # if group:
        # conditions.append({"$eq": ["$$event.group", group]})

    pipeline = [
        {"$match": {"profile_id": profile}},  # Filter to the specific profile
        {"$project": {  # Project only the calendarEvents field
            cat: {
                "$filter": {  # Filter the calendarEvents array
                    "input": f"${cat}",
                    "as": "event",
                    "cond": {  # Condition to check the date range
                        "$and": conditions
                    }
                }
            }
        }}
    ]

    # Execute the aggregation pipeline
    cursor = collection.aggregate(pipeline)
    events_list = [event for event in cursor]

    if print_events:
        # Process and print the results
        for events in events_list:
            # print(events)
            print()
            for event in events.get(cat, []):
                print(event)

    return events_list[0][cat]
def retrieve_events(name, conditions={}):
    conditions["name"] = name
    events = get_events(**conditions)
    return events

events = retrieve_events(closest_variable_name, conditions=conditions)
events

[{'name': 'Basketball Practice',
  'id': 'basketball_practice',
  'group': 'basketball_team',
  'location': 'The K Center',
  'description': 'Practice for the Duke basketball team.',
  'start_datetime': datetime.datetime(2023, 9, 3, 18, 0),
  'end_datetime': datetime.datetime(2023, 9, 3, 20, 0)},
 {'name': 'Basketball Practice',
  'id': 'basketball_practice',
  'group': 'basketball_team',
  'location': 'The K Center',
  'description': 'Practice for the Duke basketball team.',
  'start_datetime': datetime.datetime(2023, 9, 4, 18, 0),
  'end_datetime': datetime.datetime(2023, 9, 4, 20, 0)},
 {'name': 'Basketball Practice',
  'id': 'basketball_practice',
  'group': 'basketball_team',
  'location': 'The K Center',
  'description': 'Practice for the Duke basketball team.',
  'start_datetime': datetime.datetime(2023, 9, 5, 18, 0),
  'end_datetime': datetime.datetime(2023, 9, 5, 20, 0)},
 {'name': 'Basketball Practice',
  'id': 'basketball_practice',
  'group': 'basketball_team',
  'location'

Now, that we have our events, let's answer the original question: how many times do I have the event on average per week.

This entails the following directions:
- "operation": "average"
- "interval" : "week 

In [73]:
df= pd.DataFrame(events)
# Convert start_datetime to datetime
df['start_datetime'] = pd.to_datetime(df['start_datetime'])

# Set start_datetime as index
df.set_index('start_datetime', inplace=True)

# retain a single column for each unique date
df = df[["name"]]


# Resample by week and count the number of events
weekly_event_counts = df.resample('W').count()

# Calculate the average number of events per week
average_events_per_week = weekly_event_counts.mean()

# The average number of events per week
print(average_events_per_week.round(0).astype(int).values[0])

# ADD start_time and end_time to conditions
# ADD start_date and end_date to conditions

5


# Question 2

Now let's consider: `Why do I have elbow pain?`

Now, this is a hard question to answer. The final implementation of this will be a sophisticated agent. For now, it will be a little less sophisticated. It will work like this. 
- extract info from all events (in background, even before query)
- correlate what is needed, and as possible
- call upon the correlations when determining if a relationship exists
It could be structured like this:
{'elbow_pain',
'corr_with':
    'valid':false
    {
        'basketball_game':
            'corr':
                {
                    'value':
                    'method':
                }
            'caus':
                {
                    'value':
                    'method':'LLM'

                }
            'timescale':
                {'D':
                    {
                        'full_range':
                        '+1d':
                        '+2d'
                        ...
                        '
                    }
                'W':
                }
    }
}

In order to get a response for the user, the following needs to be known for initial data extraction:
- elbow_pain (correlation with all other variables)

Let's set about the business of getting proper correlation/causation with all the other variables

The first thing we need to do is determine how to assign relationships with the other variables. Right now, everything we have is a binary variable so it can all be properly compared.

# I'll want to convert all of the labesl to snake case

In [83]:
rel_dict_template = {'x':{
        'representations': {
            'binary':{
                'rel_with':
                    {
                        'y':{
                            'representations':{
                                'binary':{
                                    'valid': True,
                                    'caus':
                                        {
                                            'value': 4,
                                            'method':'LLM'

                                        },
                                    'corr':{
                                        'timescale':{
                                            'D':{
                                                'full_range':{
                                                    'corr':
                                                        {
                                                            'value': 2,
                                                            'method': 'spearman'
                                                        }
                                                }}}
                                    }
                                }
                    }
            }}
        }}
    }
    }
        

In [82]:
rel_dict_template.keys()

dict_keys(['elbow_pain'])

In [87]:
pd.DataFrame(rel_dict_template['x'])

,representations
binary,{'rel_with': {'y': {'representations': {'binar...


In [85]:
rel_dict_template['x'].keys()

dict_keys(['representations'])

In [80]:
rel_dict = {k:{} for k in schema.keys()}

In [77]:
rel_dict

{'elbow pain': {},
 'Basketball Game': {},
 'Computer Programming': {},
 'Basketball Practice': {},
 'Discrete Mathematics': {},
 'Discrete Mathematics HW due': {},
 'Computer Programming HW due': {}}